# 02 - Refusal direction on Gemma-2-2b-it (Colab / Kaggle T4)

Same pipeline as `01_qwen_local.ipynb`, run on a free T4 because Gemma-2 overflows in fp16 and
does not fit a 6 GB card in fp32.

**Before running (Colab):**

1. *Runtime > Change runtime type > T4 GPU.*
2. Accept the Gemma licence on Hugging Face (https://huggingface.co/google/gemma-2-2b-it) with the account whose token you use.
3. Add that token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and enable notebook access. On Kaggle, add it as a secret with the same name, or paste it when `huggingface_hub.login()` prompts.
4. Run all cells. Expect roughly 25-40 minutes for the 26-layer sweep plus about 45 minutes for the extensions (42 minutes measured on a T4; see the dtype note below).

**After running:** download the zip produced by the last cell, unpack it into `results/gemma-2-2b-it/` in the repo (it contains the `extensions/` folder too), and commit it; then update the README from `summary.json`, `refusal_by_layer.csv` and `extensions/*.csv`.

Set `REFUSAL_SMOKE=1` in the environment to run this notebook on CPU with `EleutherAI/pythia-70m`
and tiny splits; that is how the cells were tested without a GPU.

In [ ]:
import os, sys, subprocess, json
SMOKE = os.environ.get("REFUSAL_SMOKE") == "1"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("colab:", IN_COLAB, "| smoke mode:", SMOKE)

In [ ]:
# Clone the repo (for data/splits_seed0.json and results/) and install the package.
# Colab ships its own torch / numpy / pandas / matplotlib; those are kept rather than
# reinstalled to the versions pinned in pyproject.toml. The versions actually used are
# written to results/<model>/summary.json.
REPO_URL = "https://github.com/ORmanius/refusal-direction-replication.git"
if IN_COLAB:
    if not os.path.isdir("refusal-direction-replication"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("refusal-direction-replication")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnsight==0.8.2", "transformers==5.22.1", "accelerate==1.15.0"], check=True)
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (REPO / "pyproject.toml").exists(), f"expected the repo root, got {REPO}"
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))
print("repo:", REPO)

In [ ]:
# Hugging Face token for the gated Gemma weights.
import re

def clean_token(raw):
    """Strip whitespace, quotes and stray punctuation a paste can pick up; None if nothing is left."""
    if raw is None:
        return None
    tok = raw.strip().strip("\"'").strip(" ,;\t\r\n")
    return tok or None

raw_env = os.environ.get("HF_TOKEN")
HF_TOKEN = clean_token(raw_env)
if raw_env and not HF_TOKEN:
    print("HF_TOKEN in the environment holds no token (only punctuation or whitespace); ignoring it.")
os.environ.pop("HF_TOKEN", None)  # never let a garbled value reach huggingface_hub
if IN_COLAB and not HF_TOKEN:
    from google.colab import userdata
    raw_secret = userdata.get("HF_TOKEN")
    HF_TOKEN = clean_token(raw_secret)
    if raw_secret and not HF_TOKEN:
        print("The Colab secret HF_TOKEN holds no token (only punctuation or whitespace). "
              "Open the key icon in the left sidebar and paste the hf_... value again.")
if HF_TOKEN and not re.fullmatch(r"hf_[A-Za-z0-9]+", HF_TOKEN):
    raise ValueError(
        "HF_TOKEN does not look like a Hugging Face token (expected hf_... with letters and digits only). "
        "Re-copy the token from https://huggingface.co/settings/tokens into the Colab secret, "
        "with no spaces, commas or line breaks."
    )
if not SMOKE and not HF_TOKEN:
    from huggingface_hub import login
    login()  # interactive prompt
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("token set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
import torch
from refusal_direction import __version__
from refusal_direction.data import get_splits
from refusal_direction.model import load_model, format_prompts
from refusal_direction.pipeline import ExperimentConfig, run_experiment, config_summary
from refusal_direction.scoring import is_refusal

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16 supported (incl. emulation):", torch.cuda.is_bf16_supported())
else:
    print("no GPU")
print("refusal_direction", __version__, "| torch", torch.__version__)

## dtype on a T4

A T4 is Turing (sm_75) like the RTX 2060: it has **no native bf16**. Recent PyTorch reports bf16 as
supported on it through emulation, which works but is slow and not what Gemma was trained on;
fp16 overflows in Gemma-2. The safe choice is **float32**: the 2.6B weights take about 10.5 GB of the
T4's 15 GB, leaving room for batches of 8 prompts. Switch `DTYPE` to `"bfloat16"` if you want to try
the faster path; the results files record which was used.

In [ ]:
DTYPE = "float32"
if SMOKE:
    MODEL, DTYPE, N_TRAIN, N_VAL, LAYERS, BATCH = "EleutherAI/pythia-70m", "float32", 8, 4, [0, 1], 4
    OUT = REPO / "results" / "_smoke_pythia-70m"
    SPLITS = REPO / "results" / "_smoke_pythia-70m" / "splits_smoke.json"
    ATTN = None
else:
    MODEL, N_TRAIN, N_VAL, LAYERS, BATCH = "google/gemma-2-2b-it", 128, 32, None, 8
    OUT = REPO / "results" / "gemma-2-2b-it"
    SPLITS = REPO / "data" / "splits_seed0.json"
    ATTN = "eager"  # recommended for Gemma-2 (attention logit soft-capping)

cfg = ExperimentConfig(
    model_name=MODEL, out_dir=OUT, dtype=DTYPE, n_train=N_TRAIN, n_val=N_VAL, seed=0,
    position=-1, max_new_tokens=64, batch_size=BATCH, layers=LAYERS,
    attn_implementation=ATTN, splits_path=SPLITS, hf_token=os.environ.get("HF_TOKEN"),
)
splits = get_splits(cfg.splits_path, cfg.n_train, cfg.n_val, cfg.seed)
print({k: len(v) for k, v in splits.items() if k.startswith("harm")})
print(json.dumps(config_summary(cfg), indent=1))  # paths shown relative to the repo

In [ ]:
bundle = load_model(cfg.model_name, dtype=getattr(torch, cfg.dtype),
                    attn_implementation=cfg.attn_implementation, hf_token=cfg.hf_token)
print(f"{bundle.name}: {bundle.n_layers} layers, d_model={bundle.d_model}, {bundle.dtype}, {bundle.device}")
if torch.cuda.is_available():
    print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
print(repr(format_prompts(bundle, [splits["harmless_val"][0]])[0]))

In [ ]:
res = run_experiment(cfg, bundle=bundle)
summary = res["summary"]
print(json.dumps({k: v for k, v in summary.items() if k != "config"}, indent=1))

In [ ]:
df = res["df"]
df.style.format({c: "{:.3f}" for c in df.columns if "rate" in c or c == "direction_norm"}).hide(axis="index")

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cfg.out_dir / "refusal_by_layer.png")))

In [ ]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]

add = load_jsonl(cfg.out_dir / "completions_harmless_addition.jsonl")
base = load_jsonl(cfg.out_dir / "completions_harmless_baseline.jsonl")
for b, a in list(zip(base, add))[:4]:
    print("PROMPT:   ", a["instruction"])
    print("baseline: ", b["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(b["completion"]))
    print("+direction:", a["completion"][:160].replace("\n", " "), "| refusal =", is_refusal(a["completion"]))
    print()
abl = load_jsonl(cfg.out_dir / "completions_harmful_ablation.jsonl")
print("per-prompt refusal flags on harmful val after ablation:",
      "".join("R" if is_refusal(x["completion"]) else "." for x in abl))

## Extensions

The same four checks as `03_extensions_qwen.ipynb`, run here on the direction just fitted (about
10 more minutes on the T4). Outputs go to `results/gemma-2-2b-it/extensions/` and into the zip.

In [ ]:
from refusal_direction.extensions import run_extensions
r, r_hat, chosen = res["r"], res["r_hat"], summary["chosen_layer"]
if SMOKE:
    ext = run_extensions(cfg, bundle, r, r_hat, chosen, splits, n_transfer=4, n_reference=4, layers=LAYERS,
                         transfer_path=REPO / "results" / "_smoke_pythia-70m" / "transfer_smoke.json")
else:
    ext = run_extensions(cfg, bundle, r, r_hat, chosen, splits)
print("runtime: %.0f s" % ext["summary"]["runtime_seconds"])

## 1. Are the per-layer directions the same direction?

Cosine similarity between the difference-in-means directions of every pair of layers. The paper's
claim predicts a block of near-identical directions across the middle layers; a direction that
only worked at one layer would show up as a single bright diagonal.

In [ ]:
from IPython.display import Image, display
EXT = cfg.out_dir / "extensions"
display(Image(filename=str(EXT / "direction_similarity.png")))
cos = ext["cosine"]
print("cosine between the chosen layer's direction and each other layer's direction:")
print(" ".join(f"{l}:{cos[chosen, l]:.2f}" for l in range(bundle.n_layers)))

## 2. Does ablation damage the model?

For each layer's direction, ablated everywhere: the KL divergence from the baseline next-token
distribution at the last prompt token on the 32 harmless validation prompts (the paper's selection
criterion), and the per-token negative log-likelihood of 64 held-out Alpaca reference answers
(disjoint from the train/val instructions, listed in `extensions/reference_pairs.json`).

In [ ]:
display(Image(filename=str(EXT / "capability_by_layer.png")))
cap = ext["capability"]
cap.style.format({"kl_mean": "{:.4f}", "kl_max": "{:.3f}", "reference_nll": "{:.4f}"}).hide(axis="index")

## 3. Token-by-token dynamics

Left: the mean projection of the residual stream (at the chosen layer) onto the refusal direction,
token by token, for the 32 harmful and 32 harmless validation prompts, with and without the
intervention. Token index 0 is the first generated token; the dotted line is the end of the prompt.
Right: the same projection at every layer for one harmless prompt, with the tokens along the
x-axis, without and with the direction added.

In [ ]:
display(Image(filename=str(EXT / "projection_dynamics.png")))

## 4. Transfer to other benchmark sets

The direction was fitted on AdvBench. Here it is applied unchanged to 64 HarmBench "standard"
behaviors, 64 JailbreakBench harmful behaviors (its AdvBench-sourced rows removed) and 64
JailbreakBench benign behaviors (harmless prompts written to look borderline, so a harder test of
the addition side than Alpaca). Sampled with seed 0, saved in `data/transfer_seed0.json`.

In [ ]:
tr = ext["transfer"]
tr.style.format({"baseline_refusal_rate": "{:.3f}", "intervention_refusal_rate": "{:.3f}"}).hide(axis="index")

## 5. Dose-response: how much of the direction is needed?

The paper uses full ablation and full addition. Here the strength is varied: `x + a*r` at the
chosen layer on the harmless validation prompts (a from 0 to 1.6), and `x - a*(x.r_hat)*r_hat` at
every layer on the harmful validation prompts (a from 0 to 1, where 1 is full ablation). For every
strength: the refusal rate, the probability the model puts on its own refusal-opening tokens at the
first generated position, and one example completion. The ablation-side example is a benign
JailbreakBench prompt that the unmodified model refuses, so no harmful completion is shown. In the
GIF each frame is a different strength: the point moves along the curve and the completion changes.

In [ ]:
display(Image(filename=str(EXT / "dose_response.gif")))
dose = ext["dose_response"]
print("refusal-opening tokens used for the probability curve:", dose["refusal_first_tokens"])
dose["rows"].style.format({"refusal_rate": "{:.3f}", "p_refusal_first_token": "{:.3f}"}).hide(axis="index")

In [ ]:
for e in dose["examples"]:
    print(f"[{e['kind']}] a = {e['coef']:.1f} | refusal = {e['refusal']}")
    print("   ", e["completion"][:200].replace("\n", " "))

## 6. Geometry at the chosen layer, and which neurons carry the difference

Left: PCA of the pooled last-token residuals after the chosen layer for the 128 + 128 training
prompts, plus the harmful prompts with the direction ablated, with the refusal direction drawn as
an arrow (the GIF rotates the camera). Right: for every layer, the top 200 MLP neurons ranked by
how differently they fire on harmful vs harmless prompts (Cohen's d), and how many of the layer's
neurons exceed |d| = 1. Neuron index has no spatial meaning in a transformer, so the only
"location" that means anything here is depth.

In [ ]:
display(Image(filename=str(EXT / "pca_clouds.gif")))
g = ext["geometry"]["summary"]
print("explained variance PC1-3:", [round(x, 3) for x in g["explained_variance_pc1_3"]])
print("|cos(refusal direction, PC1-3)|:", [round(x, 3) for x in g["abs_cos_r_hat_with_pc1_3"]])
print("mean projection on the direction:", {k: round(v, 1) for k, v in g["mean_proj_r_hat"].items()})

In [ ]:
display(Image(filename=str(EXT / "neuron_effects.png")))
ext["neuron_counts"].style.format({"max_abs_d": "{:.2f}", "mean_abs_d": "{:.3f}"}).hide(axis="index")

## 7. Selecting the layer the paper's way

The main run picks the layer by validation refusal rates alone. The paper also requires the
ablated model's next-token distribution on harmless prompts to stay close to the baseline
(KL < 0.1 nats). Applying that filter to the sweep above: if it selects a different layer, the
transfer check, the dynamics figure and the dose-response curves are repeated at that layer.

In [ ]:
kl_sel = ext["kl_selected"]
print(json.dumps({k: v for k, v in kl_sel.items() if k not in ("transfer", "example_refusal")}, indent=1))
if "transfer" in kl_sel:
    import pandas as pd
    display(pd.DataFrame(kl_sel["transfer"]).style.format(
        {"baseline_refusal_rate": "{:.3f}", "intervention_refusal_rate": "{:.3f}"}).hide(axis="index"))
    display(Image(filename=str(EXT / f"projection_dynamics_layer{kl_sel['layer']}.png")))
    display(Image(filename=str(EXT / f"dose_response_layer{kl_sel['layer']}.png")))
else:
    print("the KL filter keeps the layer chosen by the main run")

In [ ]:
print(json.dumps({k: v for k, v in ext["summary"].items() if k not in ("direction_cosine_chosen_vs_neighbours", "kl_selected")}, indent=1))

In [ ]:
# Package the results for the repo. Completions of the ablated model on harmful prompts
# (completions_*_ablation*.jsonl) and directions.pt are left out on purpose.
import zipfile
zip_path = REPO / f"{cfg.out_dir.name}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(cfg.out_dir.rglob("*")):
        if p.is_file() and "_ablation" not in p.name and p.name != "directions.pt":
            z.write(p, arcname=f"{cfg.out_dir.name}/{p.relative_to(cfg.out_dir).as_posix()}")
print("wrote", zip_path, zip_path.stat().st_size // 1024, "KB")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))